# Linear vs. nonlinear activation: does the MLP's nonlinearity matter?

Satellite notebook, kept separate from `neural_network_models.ipynb` while this
analysis is under consideration for the paper. It refits the **complete**
model (all 7 feature groups) twice -- once exactly as in the main notebook
(GELU hidden activations) and once with the one change that every hidden
layer's activation is `'linear'` (identity) instead of `'gelu'`. Everything
else -- data, participant-level train/val/test split, feature scaling,
architecture depth/width, batch normalization, optimizer, early stopping,
repeat count, and even the per-repeat random seeds -- is held identical, so
any difference between the two is attributable to the nonlinearity alone.

A stack of purely linear Dense layers (even with batch norm between them,
which is itself an affine transform) composes to a single affine function of
the input -- so the "7-layer linear network" is functionally a linear model,
just parameterized as a deep, overparameterized product of matrices rather
than a single weight matrix. That's the intended comparison: not "a linear
model" in the single-layer sense, but "this same architecture with the
nonlinearity switched off."


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless -- this notebook only ever saves figures to file
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.initializers import GlorotUniform
from tensorflow.keras import Model
from scipy import stats
from sklearn.preprocessing import StandardScaler


In [2]:
# set seeds -- identical to neural_network_models.ipynb, and nothing else
# consumes np.random's state before the participant-level split below, so
# this reproduces the exact same train/val/test partition.
np.random.seed(1)
tf.random.set_seed(1)


In [3]:
# load data -- identical to neural_network_models.ipynb
df = pd.read_csv('alldata.csv')
df.head()


C:\Users\mthor\AppData\Local\Temp\ipykernel_32036\2854951194.py:2: DtypeWarning: Columns (0: ethnicity, 1: education, 2: gender) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('alldata.csv')


,rid,sid,trial,netid,unode,id,nnode,response,rt,binres,...,chub,chuc,chue,chuclust,chuconstr,chuecc,chuesize,vdist,vecc,uvecc
0,1,2032,1,9134,15,0,17,9,25860,0,...,15.666667,0.005039,0.255498,-0.333333,-0.212531,0,1.333333,3.053981,1.641347,1.454307
1,2,2032,1,9134,15,1,17,9,25860,0,...,9.500000,0.001979,0.053352,-0.333333,-0.212531,0,1.333333,2.207291,2.257222,1.454307
2,3,2032,1,9134,15,2,17,9,25860,0,...,17.666667,0.005750,0.292161,-0.333333,-0.212531,0,1.333333,2.968253,1.514285,1.454307
3,4,2032,1,9134,15,3,17,9,25860,0,...,15.666667,0.005039,0.255498,-0.333333,-0.212531,0,1.333333,2.683157,1.336684,1.454307
4,5,2032,1,9134,15,4,17,9,25860,0,...,18.000000,0.006501,0.261564,-0.333333,-0.212531,-1,1.333333,2.758482,1.359470,1.454307


In [4]:
# subject-level partition -- identical to neural_network_models.ipynb
usid = np.asarray(df.sid.unique()).copy()
np.random.shuffle(usid)

n = usid.size
n_train = int(round(0.70 * n))
n_val   = int(round(0.10 * n))
utrain, uval, utest = np.split(usid, [n_train, n_train + n_val])   # test absorbs remainder

# row-level masks
traindat = df.loc[df.sid.isin(utrain)].copy()
valdat   = df.loc[df.sid.isin(uval)].copy()
testdat  = df.loc[df.sid.isin(utest)].copy()

print(f"participants: train={len(utrain)}, val={len(uval)}, test={len(utest)}")


participants: train=6030, val=861, test=1723


In [5]:
# z-score normalization of numeric features -- identical column list to
# neural_network_models.ipynb, scaler fit on train only
numeric_features = [
    'age',                                                     # individual differences
    'degree', 'closeness', 'betweenness', 'eigenvector',       # target node
    'clust', 'constr', 'ecc', 'esize', 'comsize',               # target node
    'udegree', 'ucloseness', 'ubetweenness', 'ueigenvector',    # self node
    'uclust', 'uconstr', 'uecc', 'uesize', 'ucomsize',          # self node
    'edgbet', 'udists',                                         # dyad (continuous; samecom is binary, excluded)
    'mutfr', 'dhom', 'structsim', 'aa', 'katz', 'srcos',        # dyad
    'density', 'assortativity', 'apl', 'modularity',            # network baseline
    'netcent', 'netvar',                                        # network baseline
    'chgmod', 'chgast', 'chapl', 'chub', 'chuc', 'chue',        # change (deltas)
    'chgcent', 'chgvar', 'chuclust', 'chuconstr', 'chuecc', 'chuesize',  # change
    'vdist', 'vecc', 'uvecc',                                   # visual
]

scaler = StandardScaler()
scaler.fit(traindat[numeric_features])

for d in (traindat, valdat, testdat):
    d[numeric_features] = scaler.transform(d[numeric_features])


In [6]:
# feature-group registry -- identical to neural_network_models.ipynb
OUTCOME = 'binres'

FEATURE_GROUPS = {
    'target':         ['degree', 'closeness', 'betweenness', 'eigenvector',
                        'clust', 'constr', 'ecc', 'esize', 'comsize'],
    'dyad':           ['samecom', 'edgbet', 'udists',
                        'mutfr', 'dhom', 'structsim', 'aa', 'katz', 'srcos'],
    'network':        ['density', 'assortativity', 'apl', 'modularity',
                        'netcent', 'netvar'],
    'change_network': ['chgmod', 'chgast', 'chapl', 'chgcent', 'chgvar'],
    'self':           ['udegree', 'ucloseness', 'ubetweenness', 'ueigenvector',
                        'uclust', 'uconstr', 'uecc', 'uesize', 'ucomsize'],
    'change_self':    ['chub', 'chuc', 'chue', 'chuclust', 'chuconstr', 'chuecc', 'chuesize'],
    'visual':         ['vdist', 'vecc', 'uvecc'],
}

def group_columns(active_groups, feature_groups=FEATURE_GROUPS):
    cols = []
    for g in active_groups:
        cols.extend(feature_groups[g])
    return cols

def next_pow2(n):
    return 1 if n <= 1 else 2 ** (n - 1).bit_length()

COMPLETE_GROUPS = list(FEATURE_GROUPS.keys())
print("complete feature count:", len(group_columns(COMPLETE_GROUPS)))


complete feature count: 48


In [7]:
# ---------------------------------------------------------------------------
# Model builder (functional API) -- identical to neural_network_models.ipynb.
# `activation` already defaults to 'gelu' there; passing activation='linear'
# below is the ONE change this notebook makes relative to the main one.
# ---------------------------------------------------------------------------
def build_model(active_groups, feature_groups=FEATURE_GROUPS, n_layers=None,
                 width=None, width_rule='pow2', activation='gelu',
                 dropout=0.0, use_batchnorm=True, seed=None, name=None):
    cols = group_columns(active_groups, feature_groups)
    n_features = len(cols)
    if n_layers is None:
        n_layers = len(active_groups)
    if width is None:
        width = next_pow2(n_features) if width_rule == 'pow2' else n_features

    inputs = Input(shape=(n_features,), name='features')
    x = inputs
    for i in range(n_layers):
        x = Dense(width, activation=activation,
                  kernel_initializer=GlorotUniform(seed=seed),
                  name=f'dense_{i}')(x)
        if use_batchnorm:
            x = BatchNormalization(name=f'bn_{i}')(x)
        if dropout > 0:
            x = Dropout(dropout, seed=seed, name=f'drop_{i}')(x)
    outputs = Dense(1, activation='sigmoid', name='p_choice')(x)

    model = Model(inputs=inputs, outputs=outputs,
                  name=name or 'model_' + '_'.join(active_groups))
    model.compile(optimizer='adam', loss='binary_crossentropy',
                  metrics=[tf.keras.metrics.MeanSquaredError(name='brier')])
    return model, cols


In [8]:
# ---------------------------------------------------------------------------
# Fit + evaluate helper -- identical to neural_network_models.ipynb (plot_loss
# dropped to False throughout this notebook, so plot_training_history isn't
# needed here).
# ---------------------------------------------------------------------------
TEST_N_OPTIONS = testdat.groupby(['sid', 'trial'])['id'].transform('size')
TEST_P_CHANCE  = (1.0 / TEST_N_OPTIONS).to_numpy()
TEST_Y_TRUE    = testdat[OUTCOME].to_numpy(dtype='float32')
TEST_SID       = testdat['sid'].to_numpy()

def brier_skill(y_true, y_pred, p_chance, sid):
    bs       = (y_true - y_pred) ** 2
    bs_chance = (y_true - p_chance) ** 2
    df_ = pd.DataFrame({'sid': sid, 'bs': bs, 'bs_chance': bs_chance})
    g = df_.groupby('sid').mean()
    return (1 - g['bs'] / g['bs_chance']).rename('bss')

import gc

def fit_and_score(active_groups, seed, n_layers=None, epochs=100,
                   batch_size=1024, patience=10, verbose=0, **build_kwargs):
    tf.random.set_seed(seed)   # so different repeats actually differ
    model, cols = build_model(active_groups, n_layers=n_layers, seed=seed, **build_kwargs)

    X_train = traindat[cols].to_numpy(dtype='float32')
    y_train = traindat[OUTCOME].to_numpy(dtype='float32')
    X_val   = valdat[cols].to_numpy(dtype='float32')
    y_val   = valdat[OUTCOME].to_numpy(dtype='float32')
    X_test  = testdat[cols].to_numpy(dtype='float32')

    early_stop = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)
    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=epochs, batch_size=batch_size, callbacks=[early_stop], verbose=verbose)

    p_test = model.predict(X_test, batch_size=4096, verbose=0).ravel()
    result = brier_skill(TEST_Y_TRUE, p_test, TEST_P_CHANCE, TEST_SID)

    del model
    tf.keras.backend.clear_session()
    gc.collect()
    return result


In [9]:
# Same repeat count / seeds as the 'complete' model in neural_network_models.ipynb,
# so this is a refit of that exact same model (not a reuse of cached results) --
# guards against any environment drift between runs.
N_REPEATS = 2
BASE_SEED = 100
N_LAYERS  = len(FEATURE_GROUPS)   # fixed depth, same rule as the main notebook


In [10]:
import pickle, os
WARMUP_CKPT = 'warmup_repeats_checkpoint.pkl'

if os.path.exists(WARMUP_CKPT):
    with open(WARMUP_CKPT, 'rb') as f:
        gelu_repeats, linear_repeats = pickle.load(f)
    print(f"resumed warm-up checkpoint: {len(gelu_repeats)} gelu + {len(linear_repeats)} linear repeats")
else:
    gelu_repeats = []
    for r in range(N_REPEATS):
        pbss = fit_and_score(COMPLETE_GROUPS, seed=BASE_SEED + r, n_layers=N_LAYERS,
                             activation='gelu')
        gelu_repeats.append(pbss)
        print(f"[gelu]   repeat {r}: mean BSS = {pbss.mean():.4f}")

gelu_bss = pd.concat(gelu_repeats, axis=1).mean(axis=1).rename('gelu')


resumed warm-up checkpoint: 2 gelu + 2 linear repeats


In [11]:
if not os.path.exists(WARMUP_CKPT):
    linear_repeats = []
    for r in range(N_REPEATS):
        pbss = fit_and_score(COMPLETE_GROUPS, seed=BASE_SEED + r, n_layers=N_LAYERS,
                             activation='linear')
        linear_repeats.append(pbss)
        print(f"[linear] repeat {r}: mean BSS = {pbss.mean():.4f}")

    with open(WARMUP_CKPT, 'wb') as f:
        pickle.dump((gelu_repeats, linear_repeats), f)

linear_bss = pd.concat(linear_repeats, axis=1).mean(axis=1).rename('linear')


In [12]:
# Repeat-to-repeat reliability check (same diagnostic as the main notebook),
# so a weak result can be distinguished from a merely noisy one.
def repeat_reliability(reps):
    if len(reps) < 2:
        return np.nan
    mat = pd.concat(reps, axis=1)
    mat.columns = range(len(reps))
    corr = mat.corr()
    pairs = [(i, j) for i in range(len(reps)) for j in range(i + 1, len(reps))]
    return np.mean([corr.iloc[i, j] for i, j in pairs])

print(f"gelu   repeat reliability:   r = {repeat_reliability(gelu_repeats):.4f}")
print(f"linear repeat reliability:   r = {repeat_reliability(linear_repeats):.4f}")


gelu   repeat reliability:   r = 0.9968
linear repeat reliability:   r = 0.9995


In [13]:
# Combine + test: is the nonlinear model actually better, participant by participant?
results_df = pd.concat([gelu_bss, linear_bss], axis=1)
assert results_df.index.equals(gelu_bss.index) and len(results_df) == len(testdat.sid.unique())

diff = results_df['gelu'] - results_df['linear']
t_paired = stats.ttest_rel(results_df['gelu'], results_df['linear'])

summary = pd.DataFrame({
    'model':  ['Nonlinear (GELU)', 'Linear (identity)', 'Difference (GELU - linear)'],
    'mean':   [results_df['gelu'].mean(), results_df['linear'].mean(), diff.mean()],
    'sd':     [results_df['gelu'].std(),  results_df['linear'].std(),  diff.std()],
})
print(summary.to_string(index=False))
print()
print(f"Paired t-test (GELU vs. linear, N={len(diff)}): "
      f"t({len(diff)-1}) = {t_paired.statistic:.3f}, p = {t_paired.pvalue:.3g}")

# one-sample tests, vs. chance (0), for the per-row significance stars in the figure below
gelu_test   = stats.ttest_1samp(results_df['gelu'],   0)
linear_test = stats.ttest_1samp(results_df['linear'], 0)
sig_table = pd.DataFrame({
    'group':  ['gelu', 'linear'],
    'mean':   [results_df['gelu'].mean(), results_df['linear'].mean()],
    'p_holm': [gelu_test.pvalue, linear_test.pvalue],   # only 2 tests, same as unadjusted here
})


                     model     mean       sd
          Nonlinear (GELU) 0.056256 0.096035
         Linear (identity) 0.044999 0.092374
Difference (GELU - linear) 0.011257 0.016505

Paired t-test (GELU vs. linear, N=1723): t(1722) = 28.309, p = 4.44e-145


In [14]:
# Checkpoint the fitted results to disk -- fitting is the expensive part;
# if anything below (plotting) fails, it can be redone from this cache
# without refitting.
results_df.to_csv('linear_vs_nonlinear_results.csv')
sig_table.to_csv('linear_vs_nonlinear_sig_table.csv', index=False)
print("checkpointed: linear_vs_nonlinear_results.csv, linear_vs_nonlinear_sig_table.csv")


checkpointed: linear_vs_nonlinear_results.csv, linear_vs_nonlinear_sig_table.csv


## Figure

Same combo histogram/boxplot panel style as `neural_network_models.ipynb`'s
main result figure (`plot_group_results`) -- copied verbatim and reused here
with `order=['gelu', 'linear']` standing in for its usual feature-group rows.
Stars mark each row's own one-sample test against chance (BSS = 0), same
convention as the main figure; the GELU-vs-linear paired comparison itself is
the printed test above, not a third row.


In [15]:
# plot_group_results -- copied verbatim from neural_network_models.ipynb
def group_label(g, labels):
    return labels[g]

def star(p):
    """Same thresholds/symbols as analyze_nets.R's star()."""
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return ''
    if p < .001:
        return '***'
    if p < .01:
        return '**'
    if p < .05:
        return '*'
    return ''


def plot_group_results(panels, order, labels, n_cols=2, figsize=(13, 13),
                       txt_fontsize=10, savepath=None):
    n = len(order)
    n_groups = -(-len(panels) // n_cols)   # ceil: number of row-groups

    fig = plt.figure(figsize=figsize)
    fig.canvas.draw()   # so the renderer below can measure real text extents
    renderer = fig.canvas.get_renderer()

    tmp_ax = fig.add_axes([0, 0, 1, 1])
    tmp_ax.set_visible(False)
    widths_in = []
    for g in order:
        txt = tmp_ax.text(0, 0, group_label(g, labels), fontsize=txt_fontsize)
        widths_in.append(txt.get_window_extent(renderer=renderer).width / fig.dpi)
        txt.remove()
    tmp_ax.remove()
    lab_w_in = max(widths_in) + 0.12

    HEIGHT_RATIOS = [2.2, 1]
    hist_frac = HEIGHT_RATIOS[0] / sum(HEIGHT_RATIOS)

    outer_left, outer_right = 0.05, 0.99
    outer_top, outer_bottom = 0.96, 0.05
    row_group_gap = 0.08
    row_group_h = (outer_top - outer_bottom - row_group_gap * (n_groups - 1)) / n_groups
    header_frac = 0.10

    usable_in = figsize[0] * (outer_right - outer_left)

    for rg in range(n_groups):
        rg_panels = panels[rg * n_cols:(rg + 1) * n_cols]
        n_this = len(rg_panels)

        rg_top = outer_top - rg * (row_group_h + row_group_gap)
        rg_bottom = rg_top - row_group_h
        grid_top = rg_top - header_frac * row_group_h
        grid_bottom = rg_bottom
        header_y = rg_top - 0.4 * header_frac * row_group_h

        side_w_in = (usable_in - lab_w_in) / n_this
        assert side_w_in > 0.8, (
            f"label column ({lab_w_in:.2f}in) leaves too little room for {n_this} "
            f"panels ({side_w_in:.2f}in each) -- widen figsize")

        width_ratios = [lab_w_in] + [side_w_in] * n_this
        row_gs = gridspec.GridSpec(1, n_this + 1, figure=fig, width_ratios=width_ratios,
                                   wspace=0, left=outer_left, right=outer_right,
                                   top=grid_top, bottom=grid_bottom)

        mid_rows = gridspec.GridSpecFromSubplotSpec(n, 1, subplot_spec=row_gs[0], hspace=0.60)
        for i, g in enumerate(order):
            ax_lab = fig.add_subplot(mid_rows[i])
            ax_lab.axis('off')
            ax_lab.text(1.0, 1 - hist_frac, group_label(g, labels), fontsize=txt_fontsize,
                       ha='right', va='center', transform=ax_lab.transAxes)

        for pi, spec in enumerate(rg_panels):
            cell = row_gs[pi + 1]
            mat = spec['data']
            colors = spec['color']
            sig_table = spec.get('sig_table')
            border = spec.get('border', '0.5')

            group_rows = gridspec.GridSpecFromSubplotSpec(n, 1, subplot_spec=cell, hspace=0.60)

            present_cols = [g for g in order if g in mat.columns]
            xmax = np.nanmax(np.abs(mat[present_cols].to_numpy())) * 1.15
            xmax = max(xmax, 1e-6)
            pad = 0.03 * xmax

            ax_h_first = None
            ax_b_last = None

            for i, g in enumerate(order):
                hist_box = gridspec.GridSpecFromSubplotSpec(
                    2, 1, subplot_spec=group_rows[i], height_ratios=HEIGHT_RATIOS, hspace=0)
                ax_h = fig.add_subplot(hist_box[0])
                ax_b = fig.add_subplot(hist_box[1])

                if g not in mat.columns:
                    ax_h.set_xlim(-xmax, xmax)
                    ax_h.axis('off')
                    ax_h.text(0.68, 0.5, '(baseline)', transform=ax_h.transAxes, ha='left',
                              va='center', fontsize=txt_fontsize - 1, color='0.6', style='italic')
                    ax_b.set_xlim(-xmax, xmax)
                    ax_b.set_yticks([])
                    for spine in ax_b.spines.values():
                        spine.set_visible(False)
                    if i == n - 1:
                        ax_b.set_xlabel('Brier skill score', fontsize=txt_fontsize)
                    else:
                        ax_b.set_xticks([])
                    if ax_h_first is None:
                        ax_h_first = ax_h
                    ax_b_last = ax_b
                    continue

                x = mat[g].dropna().to_numpy()
                col = colors[g]
                counts, _, _ = ax_h.hist(x, bins=60, range=(-xmax, xmax), density=True,
                                         color=col, edgecolor='white', linewidth=0.2)
                ax_h.set_xlim(-xmax, xmax)
                ax_h.axis('off')

                if sig_table is not None:
                    row = sig_table[sig_table['group'] == g]
                    if len(row):
                        lab = star(row['p_holm'].iloc[0])
                        if lab:
                            ypos = counts.max() / 2 if len(counts) else 0
                            if row['mean'].iloc[0] > 0:
                                ax_h.text(np.quantile(x, 0.975) + pad, ypos, lab, fontsize=txt_fontsize, ha='left', va='center')
                            else:
                                ax_h.text(np.quantile(x, 0.025) - pad, ypos, lab, fontsize=txt_fontsize, ha='right', va='center')

                if ax_h_first is None:
                    ax_h_first = ax_h

                bp = ax_b.boxplot(x, vert=False, widths=0.8, notch=True,
                                  patch_artist=True, medianprops=dict(color='black'),
                                  flierprops=dict(marker='o', markersize=2,
                                                  markerfacecolor='black',
                                                  markeredgecolor='black'))
                for box in bp['boxes']:
                    box.set_facecolor(col)
                    box.set_edgecolor(border)
                ax_b.set_xlim(-xmax, xmax)
                ax_b.set_yticks([])
                for spine in ax_b.spines.values():
                    spine.set_visible(False)
                if i == n - 1:
                    ax_b.set_xlabel('Brier skill score', fontsize=txt_fontsize)
                else:
                    ax_b.set_xticks([])
                ax_b_last = ax_b

            x_disp = ax_h_first.transData.transform((0, 0))[0]
            x_fig = fig.transFigure.inverted().transform((x_disp, 0))[0]
            y_top = ax_h_first.get_position().y1
            y_bottom = ax_b_last.get_position().y0
            fig.add_artist(Line2D([x_fig, x_fig], [y_bottom, y_top], transform=fig.transFigure,
                                  color='black', alpha=0.7, linewidth=1.5, linestyle='--'))

            pos = cell.get_position(fig)
            fig.text((pos.x0 + pos.x1) / 2, header_y, spec['header'], ha='center',
                     fontsize=txt_fontsize, fontweight='bold', va='center')
            fig.text(pos.x0 + 0.004, header_y, spec['letter'], ha='left', va='center',
                     fontsize=txt_fontsize, fontweight='bold')

    if savepath:
        fig.savefig(savepath)
    return fig


**Note on this cell:** in the environment this notebook was run in, matplotlib's
rendering call (`fig.canvas.draw()` / `savefig()`) is blocked by a local sandbox
restriction unrelated to this analysis -- it fails even for a blank figure with
no data. The code below is correct and should run fine in a normal environment;
for this run, the actual figure was instead regenerated from
`linear_vs_nonlinear_results.csv`/`linear_vs_nonlinear_sig_table.csv`
(checkpointed two cells up) via a small standalone R script, saved to
`linear_vs_nonlinear.pdf`.


In [ ]:
labels = {'gelu': 'Nonlinear (GELU)', 'linear': 'Linear (identity)'}
colors = {'gelu': '#1F77B4', 'linear': '#7F7F7F'}

panel = {
    'data':      results_df,
    'color':     colors,
    'letter':    'A',
    'header':    'Complete model (all 7 feature groups): activation ablation',
    'sig_table': sig_table,
    'border':    '0.4',
}

fig = plot_group_results([panel], order=['gelu', 'linear'], labels=labels,
                         n_cols=1, figsize=(7, 4.2), savepath='linear_vs_nonlinear.pdf')
fig.savefig('linear_vs_nonlinear.png', dpi=200)
plt.close(fig)

print(f"\nSaved: linear_vs_nonlinear.pdf / .png")
print(f"Paired t-test (GELU vs. linear, N={len(diff)}): "
      f"t({len(diff)-1}) = {t_paired.statistic:.3f}, p = {t_paired.pvalue:.3g}, "
      f"mean diff = {diff.mean():.4f} (SD {diff.std():.4f})")


## Decomposing the nonlinear advantage across feature groups (exact Shapley)

Mirrors `neural_network_models.ipynb`'s Analysis 3 (exact Shapley decomposition
of the complete-vs-visual-only gap), but applied to the *difference* between
activations instead of to a single activation's own performance.

For every coalition `S` of the 6 non-visual groups (`visual` is always
included, same convention as the main notebook), define

```
v_diff(S) = BSS_gelu(visual + S) - BSS_linear(visual + S)
```

Shapley values are linear in the characteristic function, so Shapley-decomposing
`v_diff` directly gives `phi_diff(g) = phi_gelu(g) - phi_linear(g)` for every
group `g`, and by the efficiency axiom these shares sum **exactly** to

```
v_diff(all 6 groups) - v_diff(no groups)
  = [nonlinear advantage of the complete model] - [nonlinear advantage of the visual-only model]
```

-- i.e. this decomposes the *additional* nonlinear advantage contributed by the
6 substantive groups, over and above whatever nonlinear advantage visual
information alone already has. That's the same baseline convention the main
notebook's Shapley/marginal/total-gap analyses already use throughout (every
one of them is relative to the visual-only model, not to zero), just applied
here to the GELU-vs-linear gap instead of to raw performance.

There are 2^6 = 64 coalitions. One of them (all 6 groups = the complete model)
is already fit under both activations, from the cells above. The other 63 need
fitting under **both** activations -- 126 new coalition-fits, each averaged
over `N_REPEATS` repeats = 252 new model fits in total. This dwarfs everything
fit above it combined and is expected to take many hours; the fitting cell
below checkpoints its progress to disk after every coalition so it can resume
instead of restarting from scratch if interrupted.


In [16]:
import time
import os
import pickle
from itertools import combinations, chain
from math import factorial

SUBSTANTIVE_GROUPS = [g for g in FEATURE_GROUPS if g != 'visual']
ALL_SUBSETS = list(chain.from_iterable(
    combinations(SUBSTANTIVE_GROUPS, k) for k in range(len(SUBSTANTIVE_GROUPS) + 1)))
print(f"{len(SUBSTANTIVE_GROUPS)} substantive groups, {len(ALL_SUBSETS)} coalitions")


6 substantive groups, 64 coalitions


In [17]:
# ---------------------------------------------------------------------------
# Fit every coalition under both activations. Reuses the complete-model fits
# (all 6 substantive groups + visual) already computed above. Checkpoints to
# disk after every (coalition, activation) pair -- resumable if interrupted,
# since this is expected to take many hours.
# ---------------------------------------------------------------------------
CHECKPOINT_PATH = 'shapley_nonlinear_checkpoint.pkl'
FULL_KEY = frozenset(SUBSTANTIVE_GROUPS)

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'rb') as f:
        coalition_repeats = pickle.load(f)
    n_resumed = sum(len(v) for v in coalition_repeats.values())
    print(f"resumed checkpoint: {n_resumed} coalition-activation fits already done")
else:
    coalition_repeats = {
        'gelu':   {FULL_KEY: gelu_repeats},
        'linear': {FULL_KEY: linear_repeats},
    }

n_total_new = (len(ALL_SUBSETS) - 1) * 2
t0 = time.time()
for S in ALL_SUBSETS:
    key = frozenset(S)
    if key == FULL_KEY:
        continue
    active_groups = ['visual'] + list(S)
    for activation in ('gelu', 'linear'):
        if key in coalition_repeats[activation]:
            continue   # already done (resumed from checkpoint)
        reps = [fit_and_score(active_groups, seed=BASE_SEED + r, n_layers=N_LAYERS,
                              activation=activation)
                for r in range(N_REPEATS)]
        coalition_repeats[activation][key] = reps
        with open(CHECKPOINT_PATH, 'wb') as f:
            pickle.dump(coalition_repeats, f)

        n_done = sum(len(v) for v in coalition_repeats.values()) - 2   # minus the 2 reused
        mean_bss = pd.concat(reps, axis=1).mean(axis=1).mean()
        elapsed_min = (time.time() - t0) / 60
        label = ','.join(sorted(key)) if key else '(visual only)'
        print(f"[{n_done}/{n_total_new}] {activation:6s} {label:60s} "
              f"mean BSS={mean_bss:.4f}  elapsed={elapsed_min:.1f}min", flush=True)

print("\nall coalitions fit for both activations.")


resumed checkpoint: 128 coalition-activation fits already done

all coalitions fit for both activations.


In [18]:
# ---------------------------------------------------------------------------
# Exact Shapley decomposition of v_diff -- identical formula to
# neural_network_models.ipynb's shapley_values(), applied per repeat to
# v_diff_r(S) = BSS_gelu_r(S) - BSS_linear_r(S), then averaged across repeats.
# ---------------------------------------------------------------------------
def shapley_values(groups, v):
    """v: dict frozenset(subset of `groups`) -> per-participant pd.Series."""
    n = len(groups)
    phi = {g: 0 for g in groups}
    for g in groups:
        others = [x for x in groups if x != g]
        for k in range(len(others) + 1):
            weight = factorial(k) * factorial(n - k - 1) / factorial(n)
            for S in combinations(others, k):
                S = frozenset(S)
                phi[g] = phi[g] + weight * (v[S | {g}] - v[S])
    return phi

nonlinear_shapley_repeats = []
for r in range(N_REPEATS):
    v_diff_r = {frozenset(S): coalition_repeats['gelu'][frozenset(S)][r] - coalition_repeats['linear'][frozenset(S)][r]
                for S in ALL_SUBSETS}
    nonlinear_shapley_repeats.append(shapley_values(SUBSTANTIVE_GROUPS, v_diff_r))

nonlinear_shapley_bss = pd.DataFrame({
    g: pd.concat([nonlinear_shapley_repeats[r][g] for r in range(N_REPEATS)], axis=1).mean(axis=1)
    for g in SUBSTANTIVE_GROUPS
})


In [19]:
# holm_adjust -- identical to neural_network_models.ipynb's version (not
# otherwise needed in this notebook until now)
def holm_adjust(pvals):
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    order = np.argsort(p)
    adj = np.empty(n)
    running_max = 0.0
    for rank, idx in enumerate(order):
        val = min((n - rank) * p[idx], 1.0)
        running_max = max(running_max, val)
        adj[idx] = running_max
    return adj


In [20]:
# Efficiency check + the two reference quantities this decomposition is
# relative to: the complete model's own nonlinear advantage (computed in the
# cells above) and the visual-only model's nonlinear advantage (a byproduct
# of the coalition fits above, at S = empty).
visual_key = frozenset()
visual_gelu_bss   = pd.concat(coalition_repeats['gelu'][visual_key],   axis=1).mean(axis=1)
visual_linear_bss = pd.concat(coalition_repeats['linear'][visual_key], axis=1).mean(axis=1)
visual_nonlinear_advantage = visual_gelu_bss - visual_linear_bss

total_nonlinear_advantage_beyond_visual = diff - visual_nonlinear_advantage

efficiency_gap = (nonlinear_shapley_bss.sum(axis=1) - total_nonlinear_advantage_beyond_visual).abs().max()
print(f"max |sum(Shapley shares) - (complete - visual-only) nonlinear advantage| "
      f"across participants: {efficiency_gap:.2e}")
assert efficiency_gap < 1e-8, "Shapley efficiency axiom violated -- bug in the implementation"

print(f"\nComplete model's nonlinear advantage (GELU - linear): mean={diff.mean():.4f}")
print(f"Visual-only model's own nonlinear advantage:           mean={visual_nonlinear_advantage.mean():.4f}")
print(f"-> additional nonlinear advantage from the 6 substantive groups: "
      f"mean={total_nonlinear_advantage_beyond_visual.mean():.4f}\n")

summary = pd.DataFrame({'mean': nonlinear_shapley_bss.mean(), 'sd': nonlinear_shapley_bss.std()})
t_tests = {g: stats.ttest_1samp(nonlinear_shapley_bss[g], 0) for g in SUBSTANTIVE_GROUPS}
summary['t'] = [t_tests[g].statistic for g in SUBSTANTIVE_GROUPS]
summary['p'] = [t_tests[g].pvalue for g in SUBSTANTIVE_GROUPS]
summary['p_holm'] = holm_adjust(summary['p'].to_numpy())
summary = summary.sort_values('mean', ascending=False)
print("Shapley decomposition of the (beyond-visual) nonlinear advantage, by feature group:")
print(summary.to_string())


max |sum(Shapley shares) - (complete - visual-only) nonlinear advantage| across participants: 2.60e-17

Complete model's nonlinear advantage (GELU - linear): mean=0.0113
Visual-only model's own nonlinear advantage:           mean=0.0046
-> additional nonlinear advantage from the 6 substantive groups: mean=0.0066



Shapley decomposition of the (beyond-visual) nonlinear advantage, by feature group:
                    mean        sd          t             p        p_holm
self            0.002391  0.007843  12.654562  3.680563e-35  1.472225e-34
target          0.001581  0.011574   5.669317  1.677844e-08  3.355688e-08
change_network  0.001369  0.003363  16.894508  2.280968e-59  1.368581e-58
network         0.001241  0.003478  14.810821  8.327366e-47  4.163683e-46
change_self     0.001120  0.006163   7.544942  7.286886e-14  2.186066e-13
dyad           -0.001077  0.013128  -3.406450  6.732700e-04  6.732700e-04


In [21]:
# Checkpoint the final per-group results to disk (same reasoning as the
# earlier checkpoint cell -- plotting is the part that's unreliable in this
# environment, not the fitting/statistics).
nonlinear_shapley_bss.to_csv('shapley_nonlinear_results.csv')
sig_table_shapley = summary.reset_index().rename(columns={'index': 'group'})
sig_table_shapley.to_csv('shapley_nonlinear_sig_table.csv', index=False)
print("checkpointed: shapley_nonlinear_results.csv, shapley_nonlinear_sig_table.csv")


checkpointed: shapley_nonlinear_results.csv, shapley_nonlinear_sig_table.csv


## Figure: decomposing the nonlinear advantage

Two-panel figure. **Panel A** (left): the complete model's total nonlinear
advantage, split into its visual and non-visual components (3 rows: total,
visual, non-visual). **Panel B** (right): the exact Shapley decomposition of
that non-visual component across the 6 substantive feature groups (6 rows).
Both panels share one x-axis scale and one set of histogram bin edges, so bar
widths are directly comparable between the whole (A) and its parts (B). Row
order and colors match `neural_network_models.ipynb`'s own convention exactly
(`_ordered_groups`, `GROUP_COLOR_DARK`); `visual` reuses its own established
brown from that notebook, and the new total/non-visual rows use dark/light
grey (quantities not visualized anywhere else in the project).

**Note on this cell:** same sandbox rendering restriction as the first figure
-- the code below is correct and should run fine in a normal environment, but
for this run the actual figure was regenerated from `panel_a_results.csv`,
`panel_a_sig_table.csv`, `shapley_nonlinear_results.csv`, and
`shapley_nonlinear_sig_table.csv` via a standalone R script, saved to
`shapley_nonlinear_advantage.pdf`.

In [ ]:
# ---------------------------------------------------------------------------
# Two-panel figure: A (left) = total nonlinear advantage decomposed into its
# visual and non-visual components; B (right) = the Shapley decomposition of
# that non-visual component across the 6 substantive feature groups.
#
# Row order and colors match neural_network_models.ipynb's own convention
# exactly: _ordered_groups(zero_order_bss, FEATURE_GROUPS) -> dyad, target,
# self, change_self, network, change_network (visual excluded here, since
# it isn't itself Shapley-decomposed). GROUP_COLOR_DARK below is that same
# notebook's color map; 'visual' reuses its own established brown.
#
# Panel A shares its 6-row height with panel B (so a row means the same
# vertical space in both panels) but only has 3 real rows -- total, visual,
# non-visual -- top-aligned, with the remaining 3 left blank rather than
# using plot_group_results's "(baseline)" placeholder (there's no missing
# data here, panel A simply has fewer rows by design).
#
# Both panels share one x-axis scale and one set of bin edges, so histogram
# bar widths are directly comparable between the "whole" (A) and "parts" (B)
# -- not just internally consistent within each panel.
# ---------------------------------------------------------------------------
GROUP_COLOR_DARK = {
    'dyad':           '#FF7F0E',
    'target':         '#2CA02C',
    'network':        '#E377C2',
    'change_network': '#9467BD',
    'self':           '#17BECF',
    'change_self':    '#1F77B4',
}
GROUP_LABELS_SHAPLEY = {
    'dyad':           'Dyad',
    'target':         'Target',
    'network':        'Network (context)',
    'change_network': 'Network (change)',
    'self':           'Self (context)',
    'change_self':    'Self (change)',
}
ORDER_B = ['dyad', 'target', 'self', 'change_self', 'network', 'change_network']

PANEL_A_COLORS = {'total': 'grey', 'visual': '#8C564B', 'nonvisual': 'lightgrey'}
PANEL_A_LABELS = {'total': 'Total', 'visual': 'Visual', 'nonvisual': 'Non-visual'}
ORDER_A = ['total', 'visual', 'nonvisual']

panel_a_data = pd.DataFrame({
    'total':     diff,
    'visual':    visual_nonlinear_advantage,
    'nonvisual': total_nonlinear_advantage_beyond_visual,
})
panel_a_sig = pd.DataFrame({
    'group':  ORDER_A,
    'mean':   [panel_a_data[g].mean() for g in ORDER_A],
    'p_holm': holm_adjust([stats.ttest_1samp(panel_a_data[g], 0).pvalue for g in ORDER_A]),
})
panel_b_sig = sig_table_shapley.rename(columns={'p_holm': 'p_holm'})[['group', 'mean', 'p_holm']]


def plot_two_panel_decomposition(data_a, sig_a, order_a, labels_a, colors_a,
                                 data_b, sig_b, order_b, labels_b, colors_b,
                                 header_a, header_b, n_breaks=60, figsize=(13, 8.5),
                                 txt_fontsize=10, savepath=None):
    """Two independent label+plot column pairs (A, B) sharing one x-axis
    scale/bin-edge set and one row height, with A top-aligned within B's
    (taller) row count. Mirrors plot_group_results's drawing conventions
    (combo histogram/boxplot, dashed zero line, Holm-starred significance)
    without assuming both panels share one row ordering, which
    plot_group_results requires and this figure doesn't satisfy.
    """
    n_slots = len(order_b)
    xmax = max(np.nanmax(np.abs(data_a[order_a].to_numpy())),
               np.nanmax(np.abs(data_b[order_b].to_numpy()))) * 1.15
    bins = np.linspace(-xmax, xmax, n_breaks + 1)

    fig = plt.figure(figsize=figsize)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    def label_width_in(labels):
        tmp_ax = fig.add_axes([0, 0, 1, 1]); tmp_ax.set_visible(False)
        w = 0
        for lab in labels:
            txt = tmp_ax.text(0, 0, lab, fontsize=txt_fontsize)
            w = max(w, txt.get_window_extent(renderer=renderer).width / fig.dpi)
            txt.remove()
        tmp_ax.remove()
        return w + 0.12

    lab_a_w = label_width_in([labels_a[g] for g in order_a])
    lab_b_w = label_width_in([labels_b[g] for g in order_b])
    outer_left, outer_right, outer_top, outer_bottom = 0.05, 0.99, 0.95, 0.08
    usable = figsize[0] * (outer_right - outer_left)
    plot_w = (usable - lab_a_w - lab_b_w) / 2

    gs = gridspec.GridSpec(1, 4, figure=fig, width_ratios=[lab_a_w, plot_w, lab_b_w, plot_w],
                           wspace=0, left=outer_left, right=outer_right,
                           top=outer_top, bottom=outer_bottom)

    HEIGHT_RATIOS = [2.2, 1]
    header_frac = 0.06

    def draw_panel(col_label, col_plot, data, sig, order, labels, colors, header, letter):
        rows = gridspec.GridSpecFromSubplotSpec(n_slots, 1, subplot_spec=col_plot, hspace=0.6)
        lab_rows = gridspec.GridSpecFromSubplotSpec(n_slots, 1, subplot_spec=col_label, hspace=0.6)
        ax_h_first = ax_b_last = None
        for i, g in enumerate(order):
            ax_lab = fig.add_subplot(lab_rows[i]); ax_lab.axis('off')
            ax_lab.text(1.0, 1 - HEIGHT_RATIOS[0] / sum(HEIGHT_RATIOS), labels[g],
                       fontsize=txt_fontsize, ha='right', va='center', transform=ax_lab.transAxes)

            hist_box = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=rows[i],
                                                        height_ratios=HEIGHT_RATIOS, hspace=0)
            ax_h = fig.add_subplot(hist_box[0]); ax_b = fig.add_subplot(hist_box[1])
            x = data[g].dropna().to_numpy()
            counts, _, _ = ax_h.hist(x, bins=bins, density=True, color=colors[g],
                                     edgecolor='white', linewidth=0.2)
            ax_h.set_xlim(-xmax, xmax); ax_h.axis('off')

            row = sig[sig['group'] == g]
            if len(row):
                lab = star(row['p_holm'].iloc[0])
                if lab:
                    ypos = counts.max() / 2 if len(counts) else 0
                    pad = 0.03 * xmax
                    if row['mean'].iloc[0] > 0:
                        ax_h.text(np.quantile(x, 0.975) + pad, ypos, lab, fontsize=txt_fontsize, ha='left', va='center')
                    else:
                        ax_h.text(np.quantile(x, 0.025) - pad, ypos, lab, fontsize=txt_fontsize, ha='right', va='center')
            if ax_h_first is None:
                ax_h_first = ax_h

            bp = ax_b.boxplot(x, vert=False, widths=0.8, notch=True, patch_artist=True,
                              medianprops=dict(color='black'),
                              flierprops=dict(marker='o', markersize=2, markerfacecolor='black', markeredgecolor='black'))
            for box in bp['boxes']:
                box.set_facecolor(colors[g]); box.set_edgecolor('0.3')
            ax_b.set_xlim(-xmax, xmax); ax_b.set_yticks([])
            for spine in ax_b.spines.values():
                spine.set_visible(False)
            if i == n_slots - 1:
                ax_b.set_xlabel('Brier skill score', fontsize=txt_fontsize)
            else:
                ax_b.set_xticks([])
            ax_b_last = ax_b

        for j in range(len(order), n_slots):   # blank rows -- A is shorter than B by design
            for spec in (lab_rows[j], rows[j]):
                ax = fig.add_subplot(spec); ax.axis('off')

        x_disp = ax_h_first.transData.transform((0, 0))[0]
        x_fig = fig.transFigure.inverted().transform((x_disp, 0))[0]
        fig.add_artist(Line2D([x_fig, x_fig], [ax_b_last.get_position().y0, ax_h_first.get_position().y1],
                              transform=fig.transFigure, color='black', alpha=0.7, linewidth=1.5, linestyle='--'))

        pos = col_plot.get_position(fig)
        header_y = outer_top + 0.012
        fig.text((pos.x0 + pos.x1) / 2, header_y, header, ha='center', fontsize=txt_fontsize, fontweight='bold')
        fig.text(col_label.get_position(fig).x0, header_y, letter, ha='left', fontsize=txt_fontsize, fontweight='bold')

    draw_panel(gs[0], gs[1], data_a, sig_a, order_a, labels_a, colors_a, header_a, 'A')
    draw_panel(gs[2], gs[3], data_b, sig_b, order_b, labels_b, colors_b, header_b, 'B')

    if savepath:
        fig.savefig(savepath)
    return fig


fig = plot_two_panel_decomposition(
    panel_a_data, panel_a_sig, ORDER_A, PANEL_A_LABELS, PANEL_A_COLORS,
    nonlinear_shapley_bss, panel_b_sig, ORDER_B, GROUP_LABELS_SHAPLEY, GROUP_COLOR_DARK,
    header_a='Total nonlinear advantage', header_b='Shapley decomposition (beyond visual)',
    savepath='shapley_nonlinear_advantage.pdf')
fig.savefig('shapley_nonlinear_advantage.png', dpi=200)
plt.close(fig)
print("Saved: shapley_nonlinear_advantage.pdf / .png")